# Run Workflow

This notebook explains and checks the Snakemake workflow used by the tutorial. The workflow is the reproducible execution layer: `Snakefile` declares what each step consumes and produces, while `config.yml` records the parameters.

**Input:** `Snakefile`, `config.yml`, and the project scripts.

**Output:** generated benchmark data, MCC results, BIDS data, figures, and logs.

**Tutorial step:** Section 7.5, automate the analysis with Snakemake.

---


## 1. Load Workflow Configuration

This cell reads the active workflow settings. These values control the number of simulated subjects, epochs, models, effective-connectivity methods, and threshold used by the Snakemake rules.


In [5]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'Snakefile').exists():
    ROOT = Path.cwd().parent

config_path = ROOT / 'config.yml'
try:
    import yaml
    config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    for key in ['n_subjects', 'n_epochs', 't', 'fs', 'models', 'methods', 'percentile', 'bids_n_subjects']:
        print(f'{key}: {config[key]}')
except ImportError:
    print('PyYAML is not available in this kernel. Raw config.yml follows:\n')
    print(config_path.read_text(encoding='utf-8'))


n_subjects: 25
n_epochs: 5
t: 1000
fs: 256
models: ['random', 'henon', 'lorenz', 'sweep', 'cascadear', 'freqarlin', 'freqarnonlin']
methods: ['ADTF', 'PDC', 'DTF', 'cGC', 'PLI', 'PSI']
percentile: 75
bids_n_subjects: 5


## 2. Preview the Rule Graph

A dry run shows which rules Snakemake would execute without creating or modifying the workflow outputs. Use this before every full run.


In [6]:
import shutil
import subprocess

def run_command(args):
    result = subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed: {args}')

if shutil.which('snakemake'):
    run_command(['snakemake', '--dryrun'])
else:
    print('Snakemake is not available in this kernel. Run this on the host instead:')
    print('conda run -n snakemake-host snakemake --dryrun')


Snakemake is not available in this kernel. Run this on the host instead:
conda run -n snakemake-host snakemake --dryrun


## 3. Run the Workflow

Set `RUN_WORKFLOW = True` only when you want this notebook to launch the full workflow. Keeping it `False` makes the notebook safe to execute as documentation or in quick checks.


In [7]:
RUN_WORKFLOW = False

if RUN_WORKFLOW and shutil.which('snakemake'):
    run_command(['snakemake', '--cores', '1'])
elif RUN_WORKFLOW:
    print('Snakemake is not available in this kernel. Run this on the host instead:')
    print('conda run -n snakemake-host snakemake --cores 1')
else:
    print('Full workflow execution skipped. Set RUN_WORKFLOW = True to run snakemake --cores 1 when Snakemake is available.')


Full workflow execution skipped. Set RUN_WORKFLOW = True to run snakemake --cores 1 when Snakemake is available.


## 4. Check Expected Outputs

After a full run, these files provide the minimal evidence that the workflow completed: standardized BIDS data, simulated benchmark data, FC evaluation results, manuscript figures, and execution logs.


In [8]:
expected = [
    'data/bids/dataset_description.json',
    'data/simulated_connectivity_benchmark.npz',
    'data/mcc_benchmark.pkl',
    'figures/fig1_ground_truth.pdf',
    'figures/fig4_fc_matrix_grid.pdf',
    'logs/simulate.log',
    'logs/run_fc.log',
]

for rel_path in expected:
    path = ROOT / rel_path
    status = 'OK' if path.exists() else 'MISSING'
    print(f'{status:<8} {rel_path}')


OK       data/bids/dataset_description.json
OK       data/simulated_connectivity_benchmark.npz
OK       data/mcc_benchmark.pkl
OK       figures/fig1_ground_truth.pdf
OK       figures/fig4_fc_matrix_grid.pdf
MISSING  logs/simulate.log
MISSING  logs/run_fc.log


---

**Next:** archive the exact repository version, Docker image tag, and BIDS dataset DOI in the manuscript and repository README.
